# GeoAI Plogging: 상위 유흥주점 핫스팟 추출 및 시각화
10m 격자 기반 유흥주점 밀집도 상위 지역을 시각화합니다.

In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Point
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'sans-serif'


In [ ]:
# 1. 파일 경로 설정 및 피처 데이터 로드
base_dir = Path('../../')
gpkg_path = base_dir / 'data/processed/features_gwangju_south_korea_10m.gpkg'
poi_path = base_dir / 'data/raw/poi_gwangju_south_korea_raw.csv'

print("데이터 로딩 중...")
grid = gpd.read_file(gpkg_path)
print(f"총 격자 수: {len(grid):,}")


데이터 로딩 중...


DataSourceError: ../data/processed/features_gwangju_south_korea_10m.gpkg: No such file or directory

In [ ]:
# 2. 유흥주점(nightlife) 핫스팟 상위 10개 추출
top10 = grid.sort_values(
    by=['nightlife_count_30m', 'nightlife_count_50m', 'nightlife_count_100m'], 
    ascending=[False, False, False]
).head(10)

print("🔥 유흥주점 밀집도 상위 10개 핫스팟 🔥")
display(top10[['grid_id', 'nightlife_count_30m', 'nightlife_count_50m', 'nightlife_count_100m', 'geometry']])


In [ ]:
# 3. 맨 상위 1개 좌표 추출 및 주변 격자 시각화
top1 = top10.iloc[0]
bbox = top1.geometry.buffer(150)
surrounding_grids = grid[grid.geometry.intersects(bbox)]

# 4. 실제 POI 로드
poi_df = pd.read_csv(poi_path)
poi_df = poi_df.dropna(subset=['경도', '위도'])
poi_gdf = gpd.GeoDataFrame(
    poi_df, 
    geometry=[Point(xy) for xy in zip(poi_df['경도'], poi_df['위도'])], 
    crs='EPSG:4326'
).to_crs('EPSG:5179')

nightlife_poi = poi_gdf[poi_gdf['상권업종중분류명'].str.strip() == '주점']
surrounding_poi = nightlife_poi[nightlife_poi.geometry.intersects(bbox)]

# 5. 시각화
fig, ax = plt.subplots(figsize=(12, 12))

surrounding_grids.plot(
    column='nightlife_count_30m', ax=ax, cmap='Reds', edgecolor='gray', linewidth=0.2, alpha=0.8, legend=True
)

gpd.GeoDataFrame([top1], crs=grid.crs).plot(
    ax=ax, facecolor='none', edgecolor='blue', linewidth=4
)

surrounding_poi.plot(
    ax=ax, color='gold', edgecolor='black', marker='*', markersize=200
)

import matplotlib.patches as mpatches
import matplotlib.lines as mlines
top1_patch = mpatches.Patch(facecolor='none', edgecolor='blue', linewidth=4, label='Top 1 Hotspot')
poi_marker = mlines.Line2D([], [], color='gold', marker='*', linestyle='None', markersize=15, markeredgecolor='black', label='Real POI (Pubs)')
ax.legend(handles=[top1_patch, poi_marker], loc='upper left', fontsize=12)

plt.title('Top 1 Nightlife Hotspot & Surrounding Density', fontsize=18, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()
